In [7]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI


# This function will load all the variable from .env file and will
# will make available in os.eviron dictionary (env variables)
load_dotenv()

if os.environ.get("GEMINI_API_KEY"):
    print("API key variable exist")
else:
    print("GEMINI_API_KEY not found")

llm_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    google_api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
)

API key variable exist


# **CHAINS with Parallel Chains**

In [8]:
# TASK -1 [Prompts]

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Pleae summarize the movie in short : {input}" )
])

In [9]:
# TASK - 2 [LLM]

llm_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    google_api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
)

In [10]:
# TASK - 3 [Str parser]

from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [11]:
# TASK - 4 [Custom Runnable]

from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str)-> dict:

    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

# **Parallel Chain 1**

In [13]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

# TASK - 2 [LLM]

llm_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    google_api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
)

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_gemini | str_parser

# **Parallel Chain 2**

In [18]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

def insta_chain(text:dict):

    text = text["text"]

    #Task -1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a LinkedIn post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    # TASK - 2 [LLM]
    llm_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    google_api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
    )

     # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_gemini | str_parser

    result = chain_insta.invoke(text)

    return result


insta_chain_runnable = RunnableLambda(insta_chain)
    

# **Final Orchestration**

In [19]:
final_chain = (
    prompt_template | 
    llm_gemini | 
    str_parser | 
    dictionary_maker_runnable |
    RunnableParallel(branches = {"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)

In [20]:
final_chain.invoke("KGF")

{'branches': {'linkedin': 'Here are a few options for a LinkedIn post, depending on the "vibe" you want to project (Leadership, Strategy, or Personal Growth).\n\n### Option 1: The Leadership Angle (Focus on Purpose)\n**Headline: From Mercenary to Leader: The Evolution of Rocky Bhai ⚒️**\n\nIn the movie **KGF: Chapter 1**, we see a powerful transformation that every professional can learn from.\n\nRocky starts with a singular, selfish goal: to fulfill a promise to his mother to become the richest and most powerful man in the world. He takes a "hit-job" to assassinate a tyrant, purely for personal gain.\n\nBut something happens when he enters the Kolar Gold Fields. \n\nHe sees the suffering. He sees the oppression. And his mission shifts. \nHe realizes that **true power isn\'t just about how much you own—it’s about how many people you empower.**\n\nBy the end, he isn\'t just a mercenary; he’s a savior. He didn\'t just kill a villain; he liberated a workforce.\n\n**The Lesson:** You might

# **Chain as a Runnable**

In [ ]:
# TASK -1 [Beutify Function]

def beutify(final_response:dict)->dict:

    linkedin_response = final_response["branches"]["linkedin"]
    insta_response = final_response["branches"]["instagram"]

    return({"linkedin": linkedin_response, "instagram": insta_response})

beutify_runnable = RunnableLambda(beutify)

# Task 2 
#fianl_chain

beautified_chain = final_chain | beutify_runnable

beautified_chain.invoke("pushpa")